In [1]:
import sympy as sp
import numpy as np
from importlib import reload  # Python 3.4+
import ISSWlib as IS; reload(IS)

<module 'ISSWlib' from '/home/chemistry/LAI_in_snow/ISSW/ISSWlib.py'>

### Symbolic manipulations
Symbol "f" here was formerly designated as $\tau'$.

In [2]:
R1R2, chi = sp.symbols('R1R2 chi ')
top = (1-R1R2)*sp.exp(chi)+sp.sqrt((1-R1R2)**2*sp.exp(2*chi)+4*R1R2)
f = 1/2*sp.log(top/2)    
fp = sp.diff(f,R1R2,1)
fpp = sp.diff(f,R1R2,2)

print('f = ', f, '\n')
print('fp = ', fp, '\n')
print('fpp = ', fpp, '\n')
# sp.pprint(f)
# sp.pprint(fp)
# sp.pprint(fpp)

f =  0.5*log((1 - R1R2)*exp(chi)/2 + sqrt(4*R1R2 + (1 - R1R2)**2*exp(2*chi))/2) 

fp =  0.5*(-exp(chi)/2 + ((2*R1R2 - 2)*exp(2*chi)/2 + 2)/(2*sqrt(4*R1R2 + (1 - R1R2)**2*exp(2*chi))))/((1 - R1R2)*exp(chi)/2 + sqrt(4*R1R2 + (1 - R1R2)**2*exp(2*chi))/2) 

fpp =  -0.5*((exp(chi) - ((R1R2 - 1)*exp(2*chi) + 2)/sqrt(4*R1R2 + (R1R2 - 1)**2*exp(2*chi)))**2/((R1R2 - 1)*exp(chi) - sqrt(4*R1R2 + (R1R2 - 1)**2*exp(2*chi))) + (exp(2*chi) - ((R1R2 - 1)*exp(2*chi) + 2)**2/(4*R1R2 + (R1R2 - 1)**2*exp(2*chi)))/sqrt(4*R1R2 + (R1R2 - 1)**2*exp(2*chi)))/((R1R2 - 1)*exp(chi) - sqrt(4*R1R2 + (R1R2 - 1)**2*exp(2*chi))) 



### Numerical tests

In [3]:
# Optical constants for fullerene
beta1 = 8.9 # This is in m^2/g, for 450
AAE_std = 1.09; print('AAE from betas = ', AAE_std)
beta2 = beta1 * (450/600)**AAE_std; print('beta2 from AAE = ', beta2)

# Loading values on samples and their optical depths
L_range = np.array([2, 5, 10, 25, 40, 66]) # This is in micrograms
print(np.shape(L_range))
tau_range_450 = L_range*beta1/100
tau_range_600 = L_range*beta2/100
N_L_range = np.size(L_range); print(N_L_range)

# Assumed values of R1R2
R1R2_450 = 0.5
R1R2_600 = 0.6

# Assumed valueas of kappa
kappa_450 = 0.7
kappa_600 = 0.8

# f-values (formerly designated as tau')
f_range_450 = tau_range_450 / kappa_450; print('f_range_450 = ', f_range_450)
f_range_600 = tau_range_600 / kappa_600; print('f_range_600 = ', f_range_600)

# Simulating ISSW absorption optical depths
chi_range_450 = IS.get_chi_theory(f_range_450,R1R2_450); print('chi_range_450 = ', chi_range_450)
chi_range_600 = IS.get_chi_theory(f_range_600,R1R2_600); print('chi_range_600 = ', chi_range_600)

# Adding noise to the simulated ISSW absorption optical depths
noiselevel = 0.02
noise = np.random.normal(0,noiselevel,N_L_range); #print(noise)
chi_range_450 += noise; print('chi_range_450 = ', chi_range_450)
noise = np.random.normal(0,noiselevel,N_L_range); #print(noise)
chi_range_600 += noise; print('chi_range_600 = ', chi_range_600)

AAE from betas =  1.09
beta2 from AAE =  6.504393149544957
(6,)
6
f_range_450 =  [0.25428571 0.63571429 1.27142857 3.17857143 5.08571429 8.39142857]
f_range_600 =  [0.16260983 0.40652457 0.81304914 2.03262286 3.25219657 5.36612435]
chi_range_450 =  [ 1.0022753   1.92446112  3.2329073   7.05028853 10.86457575 17.47600432]
chi_range_600 =  [ 0.86595855  1.60375702  2.51890186  4.98135978  7.42068254 11.64853943]
chi_range_450 =  [ 1.03541549  1.96389711  3.23215684  7.03469304 10.86801893 17.48864249]
chi_range_600 =  [ 0.86700944  1.58305787  2.5386611   4.99666916  7.40288405 11.67002966]


### Now for a simulated retrieval

In [5]:
# Simulating deviations of R1R2 from the correct values
niter = 10

print('\nAt 450 nm ...')
delta_R1R2_450_start = -0.06; R1R2_450_start = R1R2_450 + delta_R1R2_450_start
print('starting with R1R2 = ', R1R2_450_start, 'and noise =', noiselevel)
R2R2_450, kappa_450 = IS.get_R1R1_and_kappa(chi_range_450,tau_range_450,R1R2_450_start, niter)
print('I get kappa, R1R2 = ', kappa_450, R2R2_450) 

print('\nAt 600 nm ...')
delta_R1R2_600_start = 0.04; R1R2_600_start = R1R2_600 + delta_R1R2_600_start
print('starting with R1R2 = ', R1R2_600_start, 'and noise =', noiselevel)
R2R2_600, kappa_600 = IS.get_R1R1_and_kappa(chi_range_600,tau_range_600,R1R2_600_start, niter)
print('I get kappa, R2R2 = ', kappa_600, R2R2_600) 


At 450 nm ...
starting with R1R2 =  0.44 and noise = 0.02
I get kappa, R1R2 =  0.4943503930931811 0.6980717056504034

At 600 nm ...
starting with R1R2 =  0.64 and noise = 0.02
I get kappa, R2R2 =  0.6020170432112327 0.7980746883281146
